# Check tile processing status

Live view of fleet progress, derived entirely from the Icechunk commit history via
[`global_snowmelt_runoff_onset/status.py`](../global_snowmelt_runoff_onset/status.py).
Every tile×water-year and every tile-composite refresh is one commit carrying structured
metadata (status, stats, duration, provenance); **failures never commit, so absence == not
done**. Everything here is read-only and safe to run at any time — including while a fleet
run is actively writing. Re-run the cells to refresh.

Each status call below walks the branch ancestry once, so expect a short wait per call on a
long history.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from global_snowmelt_runoff_onset.config import Config
from global_snowmelt_runoff_onset import status

In [ ]:
config = Config('config/global_config_v10.txt')
repo = config.open_output_repo()

## Overall progress

`status.get_tile_status_gdf` joins the static tile registry with the commit history: one row
per registry tile with `to_process == True`, one `wy_<year>` column per water year
(`data` / `empty` / `missing` / `ineligible` — the tile hemisphere's season hasn't elapsed
yet), a `composites` column (`data` / `empty` / `missing` / `stale`), and a rolled-up
`tile_status` (`complete` / `partial` / `unprocessed`). `complete` counts only currently
*eligible* years.

In [ ]:
tiles_gdf = status.get_tile_status_gdf(config, repo=repo)
wy_cols = [c for c in tiles_gdf.columns if c.startswith('wy_')]
tiles_gdf.head(3)

In [ ]:
n_eligible = int((tiles_gdf[wy_cols] != 'ineligible').sum().sum())
n_done = int(tiles_gdf[wy_cols].isin(['data', 'empty']).sum().sum())

print(f'registry tiles: {len(tiles_gdf):,}')
for tile_status, count in tiles_gdf.tile_status.value_counts().items():
    print(f'  {tile_status:<12}: {count:>6,} ({count / len(tiles_gdf):.1%})')
print(f'tile-years    : {n_done:,} / {n_eligible:,} eligible committed ({n_done / max(n_eligible, 1):.1%})')
print('composites    : ' + ', '.join(f'{k}: {v:,}' for k, v in tiles_gdf.composites.value_counts().items()))

Per-water-year breakdown across all registry tiles:

In [ ]:
wy_breakdown = (
    tiles_gdf[wy_cols]
    .apply(pd.Series.value_counts)
    .T.fillna(0).astype(int)
    .reindex(columns=['data', 'empty', 'missing', 'ineligible'], fill_value=0)
)
wy_eligible_counts = wy_breakdown[['data', 'empty', 'missing']].sum(axis=1)
wy_breakdown['pct_done_of_eligible'] = np.where(
    wy_eligible_counts > 0,
    (wy_breakdown['data'] + wy_breakdown['empty']) / wy_eligible_counts * 100,
    np.nan,
).round(1)
wy_breakdown.index = [c.removeprefix('wy_') for c in wy_breakdown.index]
wy_breakdown

## Commit records — all processing metadata

`status.get_commit_records` returns one row per pipeline commit (newest first), including the
per-commit `stats` dict (valid pixels, scene/orbit counts, median temporal resolution),
`duration_s`, `missing_assets` (scenes dropped because their blobs 404'd), and the full
compute-platform `provenance` (GitHub Actions run/runner, JupyterHub user, or hostname).

In [ ]:
commits_df = status.get_commit_records(repo)
commits_df['written_at'] = pd.to_datetime(commits_df['written_at'], utc=True)
commits_df['water_year'] = commits_df['water_year'].astype('Int64')
year_commits = commits_df[commits_df.kind == status.KIND_TILE_YEAR]

print(f'{len(commits_df):,} pipeline commits: {len(year_commits):,} tile-year, '
      f'{(commits_df.kind == status.KIND_TILE_COMPOSITE).sum():,} composite')
commits_df.head()

Flatten the `stats` and `provenance` dicts into columns (`stats.*`, `prov.*`) for filtering/sorting:

In [ ]:
def _flatten(column, prefix):
    frame = pd.json_normalize(commits_df[column].map(lambda d: d or {}))
    return frame.add_prefix(prefix).set_index(commits_df.index)


commits_flat = pd.concat(
    [commits_df.drop(columns=['stats', 'provenance']),
     _flatten('stats', 'stats.'),
     _flatten('provenance', 'prov.')],
    axis=1,
)
year_flat = commits_flat[commits_flat.kind == status.KIND_TILE_YEAR]
commits_flat.head()

Per-outcome summary of the tile-year commits (durations, pixel/scene stats):

In [ ]:
agg_spec = {
    'n': ('snapshot_id', 'count'),
    'median_duration_s': ('duration_s', 'median'),
    'max_duration_s': ('duration_s', 'max'),
}
for name, column in [('median_valid_px', 'stats.valid_px'),
                     ('median_scenes', 'stats.n_scenes'),
                     ('median_orbits', 'stats.n_orbits'),
                     ('median_tr_days', 'stats.median_tr_days')]:
    if column in year_flat.columns:
        agg_spec[name] = (column, 'median')

year_flat.groupby(['status', 'empty_reason'], dropna=False).agg(**agg_spec)

## Bulk processing stats → results CSV

Persists the manuscript Sect. 2.2.3 bulk-processing numbers, derived entirely from the commit
history: per-water-year and TOTAL rows with commit counts, wall time, **CPU-core-hours**
(`duration_s × prov.cpu_count`), **data volume read at the 80 m overview level** (`stats.dest_gb`),
throughput, peak memory, dask worker usage, and platform mix →
`results/<version>/bulk_processing_stats.csv`.

All pipeline commits are counted — including superseded reprocesses — so this is the fleet's *true
compute cost*, not just the cost of the surviving dataset. Commits from before 2026-08-04 predate
the `dest_gb`/`cpu_count` metadata additions; the `n_missing_*` coverage columns say how many rows
each total is missing.

In [ ]:
from global_snowmelt_runoff_onset.results import save_result_table


def _col(df, name):
    return df[name] if name in df.columns else pd.Series(np.nan, index=df.index)


def _bulk_row(scope, df):
    dur_h = df['duration_s'] / 3600
    core_h = dur_h * _col(df, 'prov.cpu_count')
    dest_gb = _col(df, 'stats.dest_gb')
    per_tile_core_h = core_h.groupby([df['row'], df['col']]).sum()
    platforms = _col(df, 'prov.platform').value_counts()
    return {
        'scope': scope,
        'n_commits': len(df),
        'n_data': int((df['status'] == status.STATUS_DATA).sum()),
        'n_empty': int((df['status'] == status.STATUS_EMPTY).sum()),
        'n_tiles': int(df[['row', 'col']].drop_duplicates().shape[0]),
        'wall_hours': round(float(dur_h.sum()), 1),
        'core_hours': round(float(core_h.sum()), 1),
        'median_core_hours_per_tile': (round(float(per_tile_core_h.median()), 2)
                                       if core_h.notna().any() else None),
        'dest_tb_read': round(float(dest_gb.sum()) / 1e3, 3),
        'mean_mb_s_effective': (round(float(_col(df, 'stats.mb_s_effective').mean()), 1)
                                if _col(df, 'stats.mb_s_effective').notna().any() else None),
        'max_peak_rss_gb': (round(float(_col(df, 'stats.peak_rss_gb').max()), 1)
                            if _col(df, 'stats.peak_rss_gb').notna().any() else None),
        'n_missing_dest_gb': int(dest_gb.isna().sum()),
        'n_missing_cpu_count': int(_col(df, 'prov.cpu_count').isna().sum()),
        'first_commit': str(df['written_at'].min()),
        'last_commit': str(df['written_at'].max()),
        'platforms': ' / '.join(f'{k}:{v}' for k, v in platforms.items()) or None,
    }


bulk_rows = [_bulk_row(f'WY{int(wy)}', year_flat[year_flat['water_year'] == wy])
             for wy in sorted(year_flat['water_year'].dropna().unique())]
bulk_rows.append(_bulk_row('TOTAL_tile_years', year_flat))
bulk_rows.append(_bulk_row('TOTAL_all_commits', commits_flat))  # incl. composite commits
bulk_df = pd.DataFrame(bulk_rows)

save_result_table(bulk_df, 'bulk_processing_stats', version=config.version, results_dir='results')
bulk_df

## Dropped S1 scenes (thinned years)

`missing_assets` records Sentinel-1 scenes the STAC catalog still lists but whose RTC blobs
were gone from Azure (404 BlobNotFound) when the year was processed: the year committed
without them ("thinned") and the scene ids are kept so those years stay distinguishable from
whole ones — and can be found and reprocessed if the upstream archive is ever repaired.
Newest commit per (tile, water year) wins, mirroring
[`scripts/list_dropped_scenes.py`](scripts/list_dropped_scenes.py).

Note this only captures blobs that died *after* cataloging. Scenes that never made it into
the STAC catalog are invisible here — coverage density shows up instead via `stats.n_scenes`
and the `temporal_resolution` variable itself.

In [ ]:
newest_per_year = year_flat.drop_duplicates(subset=['row', 'col', 'water_year'], keep='first')
thinned = newest_per_year[newest_per_year.missing_assets.notna()]

dropped_scenes = (
    thinned[['row', 'col', 'water_year', 'status', 'empty_reason', 'written_at', 'missing_assets']]
    .explode('missing_assets')
    .rename(columns={'missing_assets': 'scene_id'})
    .sort_values(['row', 'col', 'water_year'])
    .reset_index(drop=True)
)
print(f'{len(dropped_scenes):,} dropped scenes across {len(thinned):,} thinned tile-years '
      f'({thinned.groupby(["row", "col"]).ngroups:,} tiles)')
dropped_scenes

## Fleet activity and throughput

Recent commit activity (and, when the fleet is GitHub Actions, how many distinct runs/runners
committed), cumulative progress over time, and a naive ETA from the last 24 h of throughput.

In [ ]:
now = pd.Timestamp.now(tz='UTC')
latest = commits_flat.written_at.max()
if pd.isna(latest):
    print('no pipeline commits yet')
else:
    print(f'most recent commit: {latest:%Y-%m-%d %H:%M UTC} '
          f'({(now - latest).total_seconds() / 3600:.1f} h ago)')
for hours in (1, 6, 24):
    recent = commits_flat[commits_flat.written_at >= now - pd.Timedelta(hours=hours)]
    line = (f'last {hours:>2} h: {len(recent):>6,} commits '
            f'({(recent.kind == status.KIND_TILE_YEAR).sum():,} tile-years)')
    if 'prov.run_id' in recent.columns and recent['prov.run_id'].notna().any():
        line += (f' | {recent["prov.run_id"].nunique():>3} Actions runs, '
                 f'{recent["prov.runner_name"].nunique():>4} runners')
    print(line)

In [ ]:
done_over_time = year_commits.sort_values('written_at')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(done_over_time.written_at, np.arange(1, len(done_over_time) + 1), lw=1.5)
axes[0].set_ylabel('cumulative tile-year commits')
per_day = done_over_time.set_index('written_at')['snapshot_id'].resample('1D').count()
axes[1].bar(per_day.index, per_day.values, color='tab:blue')
axes[1].set_ylabel('tile-year commits / day')
for ax in axes:
    ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()

In [ ]:
window_h = 24
rate_per_h = (year_commits.written_at >= now - pd.Timedelta(hours=window_h)).sum() / window_h
n_remaining = n_eligible - n_done
print(f'throughput (last {window_h} h): {rate_per_h:.1f} tile-years/hour')
if rate_per_h > 0 and n_remaining > 0:
    print(f'{n_remaining:,} eligible tile-years remaining '
          f'-> ~{n_remaining / rate_per_h / 24:.1f} days at this rate')

## Interactive map

The registry GeoDataFrame already carries the tile metadata (snow fraction, per-polarization
S1 item counts, probe dates, notes) and the derived status columns. Merge in per-tile
processing aggregates from the newest commit per (tile, water year), then `explore()`.
The tooltip shows the headline columns on hover; **click a tile for the full popup with every
column**.

In [ ]:
newest_years = year_flat.drop_duplicates(subset=['row', 'col', 'water_year'], keep='first')

agg_spec = {
    'n_year_commits': ('snapshot_id', 'count'),
    'total_duration_min': ('duration_s', lambda s: round(float(s.sum()) / 60, 1)),
    'last_commit_at': ('written_at', 'max'),
    'n_thinned_years': ('missing_assets', lambda s: int(s.notna().sum())),
}
for name, column, how in [('total_valid_px', 'stats.valid_px', 'sum'),
                          ('max_scenes_per_year', 'stats.n_scenes', 'max'),
                          ('median_tr_days', 'stats.median_tr_days', 'median')]:
    if column in newest_years.columns:
        agg_spec[name] = (column, how)
if 'prov.platform' in newest_years.columns:
    agg_spec['platforms'] = ('prov.platform',
                             lambda s: ', '.join(sorted(s.dropna().unique())))

tile_agg = newest_years.groupby(['row', 'col']).agg(**agg_spec).reset_index()

map_gdf = tiles_gdf.merge(tile_agg, on=['row', 'col'], how='left')
if 'total_valid_px' in map_gdf.columns:
    map_gdf['total_valid_px'] = map_gdf['total_valid_px'].fillna(0).astype('int64')
# folium tooltips/popups can't serialize datetimes -- stringify them
for column in map_gdf.columns:
    if pd.api.types.is_datetime64_any_dtype(map_gdf[column]):
        map_gdf[column] = map_gdf[column].dt.strftime('%Y-%m-%d %H:%M').fillna('')
map_gdf.head(3)

In [ ]:
STATUS_COLORS = {'complete': '#2e7d32', 'partial': '#f9a825', 'unprocessed': '#c62828'}
present = [s for s in STATUS_COLORS if s in set(map_gdf.tile_status)]
tooltip_cols = [c for c in ['row', 'col', 'tile_status', 'composites', 'n_years_done',
                            'n_years_eligible', 'percent_valid_snow_pixels',
                            'total_duration_min', 'last_commit_at', 'platforms']
                if c in map_gdf.columns]

map_gdf.explore(
    column='tile_status',
    categorical=True,
    categories=present,
    cmap=[STATUS_COLORS[s] for s in present],
    tiles='Esri.WorldImagery',
    tooltip=tooltip_cols,
    popup=True,
    style_kwds={'fillOpacity': 0.6, 'weight': 0.4},
)

Same map for a single water year — set `wy_to_map` to any year in the config:

In [ ]:
wy_to_map = int(config.water_years[-1])

WY_COLORS = {'data': '#2e7d32', 'empty': '#9e9e9e', 'missing': '#c62828',
             'ineligible': '#64b5f6'}
wy_column = f'wy_{wy_to_map}'
present = [s for s in WY_COLORS if s in set(map_gdf[wy_column])]

map_gdf.explore(
    column=wy_column,
    categorical=True,
    categories=present,
    cmap=[WY_COLORS[s] for s in present],
    tiles='Esri.WorldImagery',
    tooltip=['row', 'col', wy_column, 'hemisphere', 'tile_status', 'n_years_done'],
    popup=True,
    style_kwds={'fillOpacity': 0.6, 'weight': 0.4},
)

## Remaining work

`status.get_remaining_work` is exactly what the dispatchers (`get_tiles_for_batch.py`,
`run_tiles.py`) consume: snowiest-first work items with each tile's missing water years. An
empty `water_years` list means only the composites need (re)computing.

In [ ]:
work = status.get_remaining_work(config, repo=repo)
n_years_remaining = sum(len(item['water_years']) for item in work)
composites_only = sum(1 for item in work if not item['water_years'])
print(f'{len(work):,} tiles still need work: {n_years_remaining:,} tile-years to (re)process, '
      f'{composites_only:,} tiles need only a composites refresh')
work[:5]